# Computer Vision Notebook 6: DCGAN for Face Generation with CelebA

**Goal:** understand how a generative model can learn to create image-like outputs.

A GAN has two networks:

- **Generator:** tries to create fake images.
- **Discriminator:** tries to tell real images from fake images.

They improve by competing with each other. This notebook is adapted for teaching from common DCGAN examples. Training on the full CelebA dataset is computationally heavier than the other notebooks, so a short demo may use a small subset or a pretrained checkpoint if you provide one.

**Responsible use note:** generated faces are synthetic. Do not use generated faces to impersonate real people or mislead others.

In [ ]:
!pip -q install torch torchvision matplotlib tqdm

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

## 1. Dataset options

For a live lecture, set `USE_FAKE_DATA_FOR_STRUCTURE = True` to test the code quickly. For real face generation, set it to `False` after downloading CelebA into `./data/celeba` as an image folder.

Expected folder structure for `ImageFolder`:

```text
data/celeba/img_align_celeba/000001.jpg
data/celeba/img_align_celeba/000002.jpg
...
```

You can download CelebA from Kaggle or another approved course data source, then upload or mount it in Colab.

In [ ]:
image_size = 64
batch_size = 128
USE_FAKE_DATA_FOR_STRUCTURE = True  # Change to False when CelebA images are available.

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

if USE_FAKE_DATA_FOR_STRUCTURE:
    dataset = torchvision.datasets.FakeData(size=2048, image_size=(3, image_size, image_size), transform=transform)
else:
    dataset = torchvision.datasets.ImageFolder(root='./data/celeba', transform=transform)
    # For a fast demo, use a subset. Remove this line for full training.
    dataset = Subset(dataset, range(min(5000, len(dataset))))

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

real_batch = next(iter(loader))[0][:32]
plt.figure(figsize=(8, 8))
plt.imshow(make_grid(real_batch, normalize=True).permute(1, 2, 0))
plt.axis('off')
plt.title('Training images');

## 2. Define the generator and discriminator

The generator turns random noise into an image. The discriminator outputs a probability that an image is real.

In [ ]:
z_dim = 100
feature_g = 64
feature_d = 64
channels = 3

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, feature_g*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(feature_g*8), nn.ReLU(True),
            nn.ConvTranspose2d(feature_g*8, feature_g*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g*4), nn.ReLU(True),
            nn.ConvTranspose2d(feature_g*4, feature_g*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g*2), nn.ReLU(True),
            nn.ConvTranspose2d(feature_g*2, feature_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_g), nn.ReLU(True),
            nn.ConvTranspose2d(feature_g, channels, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    def forward(self, x):
        return self.net(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, feature_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d, feature_d*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d*2), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d*2, feature_d*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d*4), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d*4, feature_d*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feature_d*8), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(feature_d*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).view(-1)

G = Generator().to(device)
D = Discriminator().to(device)
print(G)
print(D)

## 3. Train the GAN

GANs are sensitive. For a live demo, train for 1 epoch to show the loop. Better images require more data, more epochs, and GPU time.

In [ ]:
criterion = nn.BCELoss()
opt_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
opt_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
fixed_noise = torch.randn(64, z_dim, 1, 1, device=device)

num_epochs = 1
for epoch in range(num_epochs):
    for real_images, _ in tqdm(loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        real_images = real_images.to(device)
        b = real_images.size(0)

        # Train discriminator on real and fake images
        noise = torch.randn(b, z_dim, 1, 1, device=device)
        fake_images = G(noise)
        D_real = D(real_images)
        D_fake = D(fake_images.detach())
        loss_D = criterion(D_real, torch.ones_like(D_real)) + criterion(D_fake, torch.zeros_like(D_fake))
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # Train generator to fool discriminator
        D_fake_for_G = D(fake_images)
        loss_G = criterion(D_fake_for_G, torch.ones_like(D_fake_for_G))
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

    print(f'Epoch {epoch+1}: loss_D={loss_D.item():.3f}, loss_G={loss_G.item():.3f}')

## 4. Generate images

With fake data or only one epoch, outputs will not look realistic. With CelebA and more training, images gradually become face-like.

In [ ]:
G.eval()
with torch.no_grad():
    fake = G(fixed_noise).cpu()
plt.figure(figsize=(8, 8))
plt.imshow(make_grid(fake, normalize=True).permute(1, 2, 0))
plt.axis('off')
plt.title('Generated images');

## 5. Discussion

- What is the difference between classifying images and generating images?
- How could generated images be helpful for simulation, art, data augmentation, or privacy-preserving demos?
- What are the risks of realistic image synthesis?